<!-- notebook-header -->
# ARIMA e Prophet

**Modulo:** 05 - Dominios Aplicados / 05D - Series Temporais  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** ARIMA, SARIMA, Prophet, decomposicao, sazonalidade e validacao temporal.


# 5D_2: ARIMA e Prophet - Previsao de Series Temporais

**Objetivo**: Compreender dois paradigmas classicos de previsao:
ARIMA (autorregressive/statistical) e Prophet (trend + seasonality decomposition).

**Contexto**: Em 5D_1, entendemos autocorrelacao e estrutura temporal de series.
Agora respondemos a pergunta central: "como PREVER valores futuros"?
Existem duas abordagens principais:
1. **ARIMA**: Modela a SERIE como combinacao de autorregresso + media movel
2. **Prophet**: Decompoe a serie em TREND + SEASONALITY + HOLIDAYS

**Neste notebook:**
1. Intuicao classica: Prever como clima (tendencia, sazonalidade, perturbacoes)
2. Processo Autoregressivo (AR): primeira ordem, depois p-esima
3. Processo Media Movel (MA): erros passados importam
4. ARMA = combinacao de AR + MA
5. Diferenciacao e ARIMA (I = Integrated)
6. Selecao de ordem (p,d,q) via ACF/PACF
7. Previsao forward com ARIMA
8. SARIMA para sazonalidade
9. Prophet: decomposicao trend + seasonality + holidays
10. Comparacao ARIMA vs Prophet vs ML
11. Exercicios praticos
12. Erros comuns na previsao

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

## 1. Modelos Classicos de Previsao: Analogia do Clima

### Problema: Prever a Temperatura Amanha
Imagine tentar prever a temperatura em Sao Paulo amanha. Voce poderia pensar:

1. **Tendencia (Trend):** A temperatura MÉDIA aumenta de dezembro para janeiro?
2. **Sazonalidade (Seasonality):** Faz mais frio no inverno, mais quente no verao?
3. **Autocorrelacao (AR):** Se hoje fez 28°C, amanha provavelmente fara ~27-29°C (nao varia muito)?
4. **Surpresas (MA):** Se choveu ontem (erro grande), pode fazer mais frio hoje (erro passado importa)?

### Definicao Formal
Uma serie temporal Y_t pode ser decomposta como:
Y_t = Trend_t + Seasonality_t + Cyclical_t + Error_t

**ARIMA** captura: Y_t depende de valores passados + erros passados (forma parametrica)
**Prophet** captura: decomposicao explícita de trend, seasonality, holidays

### Por que em ML (Importancia Pratica)
Series temporais aparecem em TODA PARTE:
- Financas: prever precos de acoes, volatilidade, cambio
- Energia: prever consumo de eletricidade, geracao eolica/solar
- E-commerce: prever vendas mensais, demanda por produto
- Saude: prever casos de gripe, COVID, ocupacao hospitalar
- Trafico: prever numero de usuarios online, acessos servidor

Antes de usar deep learning (5D_3: LSTM), entender ARIMA/Prophet e crucial
porque eles sao:
1. **Interpretaveis:** entendemos cada componente (trend, seasonality, shocks)
2. **Eficientes:** funcionam com poucos dados (100-200 pontos, LSTM precisa 500+)
3. **Rapidos:** treinamento em segundos/minutos (LSTM minutos/horas)
4. **Baselines:** servem para comparar com ML complexo
5. **Confiáveis:** fundamentacao teorica solida (nao black-box)
6. **Expliciveis:** posso mostrar para stakeholders por que prevejo X

In [ ]:
# Exemplo visual: decomposicao da temperatura
np.random.seed(42)
t = np.arange(0, 365)

# Trend: aumento medio na primavera/verao
trend = 20 + 0.05 * t + 3 * np.sin(2 * np.pi * t / 365)

# Seasonalidade: ciclo semanal (temperatura mais alta no fim de semana?)
seasonality = 2 * np.sin(2 * np.pi * t / 7)

# Ciclo semanal adicional (variacao diaria)
intraweek = 1.5 * np.sin(2 * np.pi * t / 365)

# Noise aleatorio (perturbacoes imprevisiveis)
noise = np.random.normal(0, 0.8, len(t))

# Serie completa
Y = trend + seasonality + intraweek + noise

# Visualizar
fig, axes = plt.subplots(5, 1, figsize=(14, 10))

# 1. Serie original
axes[0].plot(t, Y, 'k-', linewidth=1, alpha=0.7)
axes[0].set_ylabel('Temperatura (°C)')
axes[0].set_title('Serie Original (Temperatura)', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# 2. Trend
axes[1].plot(t, trend, 'b-', linewidth=2)
axes[1].set_ylabel('Trend')
axes[1].set_title('Componente Trend (tendencia de longo prazo)', fontweight='bold')
axes[1].grid(True, alpha=0.3)

# 3. Seasonalidade semanal
axes[2].plot(t, seasonality, 'g-', linewidth=1)
axes[2].set_ylabel('Sazonalidade')
axes[2].set_title('Sazonalidade (ciclo semanal)', fontweight='bold')
axes[2].grid(True, alpha=0.3)

# 4. Intraweek
axes[3].plot(t, intraweek, 'orange', linewidth=1)
axes[3].set_ylabel('Ciclo Anual')
axes[3].set_title('Ciclo Anual (variacao por estacao)', fontweight='bold')
axes[3].grid(True, alpha=0.3)

# 5. Noise
axes[4].plot(t, noise, 'r-', linewidth=0.5, alpha=0.5)
axes[4].set_ylabel('Ruido')
axes[4].set_xlabel('Dias')
axes[4].set_title('Perturbacoes Aleatorias (nao previáveis)', fontweight='bold')
axes[4].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/decomposition.png', dpi=100, bbox_inches='tight')
plt.show()

print("Decomposicao de uma Serie Temporal")
print("=" * 60)
print("Y_t = Trend_t + Seasonality_t + Ciclo_t + Noise_t")
print()
print("Objetivo de previsao:")
print("- Modelar Trend, Seasonality, Ciclo (partes previsiveis)")
print("- Aceitar Noise (partes imprevisiveis)")
print()
print("ARIMA: captura autocorrelacao e erros passados")
print("Prophet: decompoe explicitamente trend + seasonality + holidays")

## 2. Processo AR (Autoregressivo): Y_t depende do passado

### Analogia
Imagine o humor de uma pessoa. Se ela esta feliz hoje, provavelmente estara
feliz amanha (inércia). Se estava triste ontem e feliz hoje, provavelmente
o humor fluctua. Processo AR: o presente depende do PASSADO.

### Definicao Formal
Processo AR(p): Y_t = c + phi_1 * Y_{t-1} + phi_2 * Y_{t-2} + ... + phi_p * Y_{t-p} + epsilon_t

Onde:
- phi_i: coeficientes autorregressive (quanto cada lag importa)
- epsilon_t: ruido branco (imprevísível)

Exemplo **AR(1)**: Y_t = c + phi_1 * Y_{t-1} + epsilon_t
- Se phi_1 = 0.9 (alto): serie muito persistente (muda devagar)
- Se phi_1 = 0.3 (baixo): serie varia rapido
- Se phi_1 = 0 (nulo): serie e ruido puro

### Por que em ML (Autocorrelacao e Core da Previsao)
AR modela a dependencia temporal fundamental: informacao do passado ajuda a prever o futuro.
E a base dos modelos autoregressivos em deep learning (5D_3: LSTM/Transformers).
Qualquer modelo de series que nao capture autocorrelacao esta deixando poder na mesa.

Em pratica: AR(1) com phi=0.9 supera ruido puro por ~90% em previsao 1-passo.
Ignorar phi e deixar ganhos faceis no chao.

In [ ]:
# Simular processos AR com diferentes coeficientes
def simulate_ar(phi, c=0, n_steps=200, noise_std=1.0):
    # Y_t = c + phi[0] * Y_{t-1} + phi[1] * Y_{t-2} + ... + noise
    p = len(phi)
    Y = np.zeros(n_steps)
    Y[:p] = np.random.normal(0, noise_std, p)

    for t in range(p, n_steps):
        Y[t] = c + np.sum(phi * Y[t-p:t][::-1]) + np.random.normal(0, noise_std)

    return Y

# 1. AR(1) com diferentes phi
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
phis = [0.1, 0.5, 0.9, -0.7]
titles = ['AR(1): phi=0.1 (baixa persistencia)',
          'AR(1): phi=0.5 (media persistencia)',
          'AR(1): phi=0.9 (alta persistencia)',
          'AR(1): phi=-0.7 (alternancia)']

for idx, (phi, title) in enumerate(zip(phis, titles)):
    ax = axes[idx // 2, idx % 2]
    Y = simulate_ar([phi], n_steps=200)
    ax.plot(Y, 'b-', linewidth=1.5)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Y_t')
    ax.set_xlabel('Tempo')
    ax.grid(True, alpha=0.3)

    # Linha media
    ax.axhline(y=np.mean(Y), color='r', linestyle='--', alpha=0.5, label=f'Media={np.mean(Y):.2f}')
    ax.legend()

plt.tight_layout()
plt.savefig('/tmp/ar_processes.png', dpi=100, bbox_inches='tight')
plt.show()

print("Processos AR(1) com diferentes coeficientes")
print("=" * 60)
print("AR(1): Y_t = phi * Y_{t-1} + epsilon_t")
print()
print("phi=0.1: serie varia muito rapido (parece ruido)")
print("phi=0.5: persistencia media")
print("phi=0.9: persistencia ALTA (muda muito devagar)")
print("phi=-0.7: oscila (sobe-desce alternadamente)")

In [ ]:
# 2. AR(2) para capturar ciclos
phi_ar2 = [1.3, -0.4]  # Gera ciclo de ~8 passos
Y_ar2 = simulate_ar(phi_ar2, n_steps=200, noise_std=0.3)

# 3. Comparar ACF (autocorrelacao) de diferentes AR
def calculate_acf(y, max_lag=20):
    y_centered = y - np.mean(y)
    acf = np.correlate(y_centered, y_centered, mode='full')
    acf = acf[len(acf)//2:]
    acf = acf / acf[0]
    return acf[:max_lag+1]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for idx, (phi, title) in enumerate([([0.7], 'AR(1): phi=0.7'),
                                      ([1.2, -0.5], 'AR(2): ciclo'),
                                      ([0.3], 'AR(1): phi=0.3')]):
    ax = axes[idx]
    Y = simulate_ar(phi, n_steps=500)
    acf = calculate_acf(Y, max_lag=20)

    ax.bar(range(len(acf)), acf, color='steelblue', alpha=0.7)
    ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    ax.axhline(y=0.05, color='r', linestyle='--', alpha=0.5)
    ax.axhline(y=-0.05, color='r', linestyle='--', alpha=0.5)
    ax.set_xlabel('Lag')
    ax.set_ylabel('ACF')
    ax.set_title(title, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/ar_acf.png', dpi=100, bbox_inches='tight')
plt.show()

print("Autocorrelacao Function (ACF) de diferentes AR")
print("=" * 60)
print("ACF mede correlacao Y_t com Y_{t-k} (lag k)")
print()
print("AR(1) com phi alta: ACF decai lentamente (exponencial)")
print("AR(2) com ciclo: ACF oscila (reflete ciclo)")
print("AR(1) com phi baixa: ACF decai rapido")

## 3. Processo MA (Media Movel): Erros passados importam

### Analogia
Imagine o preco de um produto. Se ontem teve promocao (surprise negativa),
a demanda fica altas HOJE. Surpresa (erro) passada afeta o presente.
MA: o presente depende dos ERROS passados, nao dos valores passados.

### Definicao Formal
Processo MA(q): Y_t = mu + epsilon_t + theta_1 * epsilon_{t-1} + theta_2 * epsilon_{t-2} + ... + theta_q * epsilon_{t-q}

Onde:
- epsilon_t: ruido branco em t
- theta_i: coeficientes de media movel
- mu: media

**Diferenca AR vs MA:**
- AR(p): Y_t depende de Y_{t-1}, Y_{t-2}, ...
- MA(q): Y_t depende de epsilon_{t-1}, epsilon_{t-2}, ...

### Por que em ML
MA captura "shocks" (surpresas) passadas que ainda afetam o presente.
Em economia, mercados reagem a noticias (surpresas) que afetam precos pelos dias seguintes.

In [ ]:
# Simular processos MA com diferentes coeficientes
def simulate_ma(theta, mu=0, n_steps=200, noise_std=1.0):
    # Y_t = mu + epsilon_t + theta[0] * epsilon_{t-1} + ...
    q = len(theta)
    epsilon = np.random.normal(0, noise_std, n_steps)
    Y = np.zeros(n_steps)

    for t in range(n_steps):
        Y[t] = mu
        Y[t] += epsilon[t]
        for j in range(1, q+1):
            if t >= j:
                Y[t] += theta[j-1] * epsilon[t-j]

    return Y

# MA(1) com diferentes theta
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

thetas = [0.7, 0.0, -0.7]
titles = ['MA(1): theta=0.7', 'MA(1): theta=0 (ruido puro)', 'MA(1): theta=-0.7']

for idx, (theta, title) in enumerate(zip(thetas, titles)):
    ax = axes[idx]
    Y = simulate_ma([theta], n_steps=200)
    ax.plot(Y, 'b-', linewidth=1.5)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Y_t')
    ax.set_xlabel('Tempo')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/ma_processes.png', dpi=100, bbox_inches='tight')
plt.show()

print("Processos MA(1) com diferentes coeficientes")
print("=" * 60)
print("MA(1): Y_t = epsilon_t + theta * epsilon_{t-1}")
print()
print("theta=0.7: erros passados aumentam o efeito (oscilacoes suaves)")
print("theta=0: nenhum efeito de erros (ruido puro)")
print("theta=-0.7: erros passados cancelam (meio aleatorio)")

In [ ]:
# Comparar ACF de MA vs AR
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# AR vs MA: ACF behavior
ar_process = simulate_ar([0.7], n_steps=500)
ma_process = simulate_ma([0.7], n_steps=500)
ma_process_2 = simulate_ma([0.7, 0.5], n_steps=500)

processes = [ar_process, ma_process, ma_process_2]
titles = ['AR(1): phi=0.7', 'MA(1): theta=0.7', 'MA(2): theta=[0.7,0.5]']

# ACF
for idx, (Y, title) in enumerate(zip(processes, titles)):
    ax = axes[0, idx]
    acf = calculate_acf(Y, max_lag=15)
    ax.bar(range(len(acf)), acf, color='steelblue', alpha=0.7)
    ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    ax.axhline(y=0.05, color='r', linestyle='--', alpha=0.5)
    ax.set_xlabel('Lag')
    ax.set_ylabel('ACF')
    ax.set_title(f'ACF: {title}', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

# Series
for idx, (Y, title) in enumerate(zip(processes, titles)):
    ax = axes[1, idx]
    ax.plot(Y[:100], 'b-', linewidth=1)
    ax.set_ylabel('Y_t')
    ax.set_xlabel('Tempo')
    ax.set_title(f'Serie: {title}', fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/ar_vs_ma.png', dpi=100, bbox_inches='tight')
plt.show()

print("Diferenca entre AR e MA (via ACF)")
print("=" * 60)
print("AR(p): ACF decai gradualmente (infinito)")
print("MA(q): ACF corta depois de lag q (0 apos lag q)")
print()
print("Dica: Olhe a ACF para decidir entre AR vs MA!")

## 4. ARMA: Combinando AR e MA

### Definicao Formal
ARMA(p, q) = AR(p) + MA(q):

Y_t = c + phi_1*Y_{t-1} + ... + phi_p*Y_{t-p} + epsilon_t + theta_1*epsilon_{t-1} + ... + theta_q*epsilon_{t-q}

Combina:
- AR(p): dependencia de valores passados (capturam trends)
- MA(q): dependencia de erros passados (capturam shocks)

### Exemplo ARMA(1,1)
Y_t = c + phi*Y_{t-1} + epsilon_t + theta*epsilon_{t-1}

### Por que em ML
ARMA e mais flexivel que AR ou MA sozinhos. A maioria das series reais
segue ARMA, nao AR ou MA puro.

In [ ]:
# Simular ARMA(1,1)
def simulate_arma(phi, theta, c=0, n_steps=200, noise_std=1.0):
    # Y_t = c + sum(phi_i * Y_{t-i}) + eps_t + sum(theta_j * eps_{t-j})
    p = len(phi) if phi is not None else 0
    q = len(theta) if theta is not None else 0
    max_lag = max(p, q)

    epsilon = np.random.normal(0, noise_std, n_steps)
    Y = np.zeros(n_steps)

    for t in range(max_lag, n_steps):
        Y[t] = c + epsilon[t]

        # AR terms
        for i in range(p):
            Y[t] += phi[i] * Y[t-i-1]

        # MA terms
        for j in range(q):
            Y[t] += theta[j] * epsilon[t-j-1]

    return Y

# Comparar ARMA vs AR vs MA
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# AR(1)
Y_ar = simulate_ar([0.7], n_steps=200)
axes[0, 0].plot(Y_ar, 'b-', linewidth=1.5)
axes[0, 0].set_title('AR(1): phi=0.7', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# MA(1)
Y_ma = simulate_ma([0.7], n_steps=200)
axes[0, 1].plot(Y_ma, 'g-', linewidth=1.5)
axes[0, 1].set_title('MA(1): theta=0.7', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# ARMA(1,1)
Y_arma = simulate_arma([0.5], [0.6], n_steps=200)
axes[1, 0].plot(Y_arma, 'r-', linewidth=1.5)
axes[1, 0].set_title('ARMA(1,1): phi=0.5, theta=0.6', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# ACF: ARMA
acf_arma = calculate_acf(Y_arma, max_lag=15)
axes[1, 1].bar(range(len(acf_arma)), acf_arma, color='steelblue', alpha=0.7)
axes[1, 1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[1, 1].axhline(y=0.05, color='r', linestyle='--', alpha=0.5)
axes[1, 1].set_xlabel('Lag')
axes[1, 1].set_ylabel('ACF')
axes[1, 1].set_title('ACF do ARMA(1,1)', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/arma.png', dpi=100, bbox_inches='tight')
plt.show()

print("ARMA(1,1): Combinando AR e MA")
print("=" * 60)
print("ARMA combina:")
print("- AR: captura persistencia (valores passados)")
print("- MA: captura shocks (erros passados)")
print()
print("ACF do ARMA mostra decaimento (como AR) + cutoff (como MA)")

## 5. Diferenciacao e ARIMA: Tornando series estacionarias

### Problema: Series Nao-Estacionarias
Uma serie estacionaria tem media, variancia e autocorrelacao CONSTANTES.
Exemplo NAO estacionaria: stock price (sempre sobe a longo prazo).

Sem estacionaridade, AR/MA/ARMA nao funcionam!

### Solucao: Diferenciacao
Primeira diferenca: Y_t' = Y_t - Y_{t-1}
Segunda diferenca: Y_t'' = Y_t' - Y_t-1'

Diferenciacao remove TRENDS!

### Definicao Formal ARIMA(p,d,q)
I = Integrated (diferenciacao)
- p: ordem AR
- d: numero de diferenciacoes
- q: ordem MA

ARIMA(p,d,q):
1. Diferenciar a serie d vezes: Y_t^{(d)} = (diferenca d-esima)
2. Aplicar ARMA(p,q) em Y_t^{(d)}

### Por que em ML
ARIMA e o modelo PADRAO para series temporais nao-estacionarias.
Funciona para 80% das aplicacoes praticas.

In [ ]:
# Demonstrar diferenciacao
np.random.seed(42)
t = np.arange(0, 100)

# Serie nao-estacionaria: trend + noise
trend = 0.1 * t
noise = np.random.normal(0, 1, len(t))
Y = trend + noise

# Primeira diferenca
Y_diff1 = np.diff(Y)

# Segunda diferenca
Y_diff2 = np.diff(Y_diff1)

fig, axes = plt.subplots(3, 2, figsize=(14, 10))

# Original
axes[0, 0].plot(Y, 'b-', linewidth=1.5)
axes[0, 0].set_title('Serie Original (nao-estacionaria)', fontweight='bold')
axes[0, 0].set_ylabel('Y_t')
axes[0, 0].grid(True, alpha=0.3)

acf_y = calculate_acf(Y, max_lag=20)
axes[0, 1].bar(range(len(acf_y)), acf_y, color='steelblue', alpha=0.7)
axes[0, 1].set_title('ACF: Original (decai LENTAMENTE!)', fontweight='bold')
axes[0, 1].axhline(y=0.05, color='r', linestyle='--', alpha=0.5)
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Primeira diferenca
axes[1, 0].plot(Y_diff1, 'g-', linewidth=1.5)
axes[1, 0].set_title('Primeira Diferenca: Y_t - Y_t-1', fontweight='bold')
axes[1, 0].set_ylabel('Y_t')
axes[1, 0].axhline(y=0, color='k', linestyle='--', alpha=0.5)
axes[1, 0].grid(True, alpha=0.3)

acf_diff1 = calculate_acf(Y_diff1, max_lag=20)
axes[1, 1].bar(range(len(acf_diff1)), acf_diff1, color='green', alpha=0.7)
axes[1, 1].set_title('ACF: Primeira Diferenca (melhor!)', fontweight='bold')
axes[1, 1].axhline(y=0.05, color='r', linestyle='--', alpha=0.5)
axes[1, 1].grid(True, alpha=0.3, axis='y')

# Segunda diferenca
axes[2, 0].plot(Y_diff2, 'r-', linewidth=1.5)
axes[2, 0].set_title('Segunda Diferenca', fontweight='bold')
axes[2, 0].set_ylabel('Y_t''')
axes[2, 0].axhline(y=0, color='k', linestyle='--', alpha=0.5)
axes[2, 0].grid(True, alpha=0.3)

acf_diff2 = calculate_acf(Y_diff2, max_lag=20)
axes[2, 1].bar(range(len(acf_diff2)), acf_diff2, color='red', alpha=0.7)
axes[2, 1].set_title('ACF: Segunda Diferenca (pode ser demais)', fontweight='bold')
axes[2, 1].axhline(y=0.05, color='r', linestyle='--', alpha=0.5)
axes[2, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/differencing.png', dpi=100, bbox_inches='tight')
plt.show()

print("Diferenciacao: Removendo Trend para Estacionaridade")
print("=" * 60)
print("Original: ACF decai LENTAMENTE (nao-estacionaria)")
print("d=1: ACF decai RAPIDO (estacionaria!)")
print("d=2: pode ser DEMAIS (perder informacao)")
print()
print("Dica: comece com d=1, use d=2 se necessario")

## 6. Como Escolher (p,d,q): Lendo ACF e PACF

### ACF (Autocorrelacao) e PACF (Partial Autocorrelacao)
- **ACF(k):** correlacao entre Y_t e Y_{t-k}
- **PACF(k):** correlacao entre Y_t e Y_{t-k} removendo Y_t-1, ..., Y_{t-k+1}

PACF nos ajuda a escolher p (ordem AR).

### Tabela de Decisao

| ACF | PACF | Modelo |
|-----|------|--------|
| Decai exponencialmente | Picos em lag p | AR(p) |
| Picos ate lag q | Decai | MA(q) |
| Decai + picos | Picos ate lag p | ARMA(p,q) |

### Por que em ML
Escolher (p,d,q) correto evita:
- Underfitting (modelo muito simples)
- Overfitting (modelo muito complexo)

Tipicamente: p,q <= 2, d <= 2 funciona para 90% dos casos.

In [ ]:
# Calcular PACF (Partial Autocorrelation)
def calculate_pacf(y, max_lag=20):
    # Yule-Walker simplificado
    c = calculate_acf(y, max_lag=max_lag)
    pacf = np.zeros(len(c))
    pacf[0] = 1.0

    if max_lag >= 1:
        pacf[1] = c[1]

    for k in range(2, len(c)):
        numerator = c[k] - sum(pacf[j] * c[k-j] for j in range(1, k))
        denominator = 1 - sum(pacf[j] * c[j] for j in range(1, k))
        if abs(denominator) > 1e-10:
            pacf[k] = numerator / denominator

    return pacf

# Comparar ACF/PACF para diferentes processos
fig, axes = plt.subplots(3, 2, figsize=(14, 10))

# AR(2)
Y_ar2 = simulate_ar([1.2, -0.5], n_steps=300)
acf_ar2 = calculate_acf(Y_ar2, max_lag=15)
pacf_ar2 = calculate_pacf(Y_ar2, max_lag=15)

axes[0, 0].bar(range(len(acf_ar2)), acf_ar2, color='steelblue', alpha=0.7)
axes[0, 0].axhline(y=0.05, color='r', linestyle='--', alpha=0.5)
axes[0, 0].set_title('ACF de AR(2): decai gradualmente', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3, axis='y')

axes[0, 1].bar(range(len(pacf_ar2)), pacf_ar2, color='green', alpha=0.7)
axes[0, 1].axhline(y=0.05, color='r', linestyle='--', alpha=0.5)
axes[0, 1].set_title('PACF de AR(2): 2 picos (p=2)!', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# MA(2)
Y_ma2 = simulate_ma([0.7, 0.3], n_steps=300)
acf_ma2 = calculate_acf(Y_ma2, max_lag=15)
pacf_ma2 = calculate_pacf(Y_ma2, max_lag=15)

axes[1, 0].bar(range(len(acf_ma2)), acf_ma2, color='steelblue', alpha=0.7)
axes[1, 0].axhline(y=0.05, color='r', linestyle='--', alpha=0.5)
axes[1, 0].set_title('ACF de MA(2): picos ate lag 2 (q=2)!', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

axes[1, 1].bar(range(len(pacf_ma2)), pacf_ma2, color='green', alpha=0.7)
axes[1, 1].axhline(y=0.05, color='r', linestyle='--', alpha=0.5)
axes[1, 1].set_title('PACF de MA(2): decai gradualmente', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

# ARMA(1,1)
Y_arma11 = simulate_arma([0.6], [0.5], n_steps=300)
acf_arma11 = calculate_acf(Y_arma11, max_lag=15)
pacf_arma11 = calculate_pacf(Y_arma11, max_lag=15)

axes[2, 0].bar(range(len(acf_arma11)), acf_arma11, color='steelblue', alpha=0.7)
axes[2, 0].axhline(y=0.05, color='r', linestyle='--', alpha=0.5)
axes[2, 0].set_title('ACF de ARMA(1,1): comportamento misto', fontweight='bold')
axes[2, 0].grid(True, alpha=0.3, axis='y')

axes[2, 1].bar(range(len(pacf_arma11)), pacf_arma11, color='green', alpha=0.7)
axes[2, 1].axhline(y=0.05, color='r', linestyle='--', alpha=0.5)
axes[2, 1].set_title('PACF de ARMA(1,1): comportamento misto', fontweight='bold')
axes[2, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/acf_pacf.png', dpi=100, bbox_inches='tight')
plt.show()

print("ACF e PACF: Como Escolher (p, q)")
print("=" * 60)
print("AR(2): PACF tem 2 picos significativos")
print("MA(2): ACF tem 2 picos significativos")
print("ARMA(1,1): ambos mostram comportamento misto")
print()
print("Dica: Procure por picos significativos fora da banda vermelha!")

## 7. Previsao com ARIMA: Predicao Forward

Apos estimar (p,d,q), o modelo ARIMA pode prever h passos a frente.

### Algoritmo
1. Estimar parametros phi, theta do ARMA
2. Para cada step futuro t+k:
   - Usar valores conhecidos de Y
   - Usar erros estimados do passado
   - Prever Y_{t+k} iterativamente

### Incerteza Cresce com o Horizonte
- Previsao 1 passo a frente: muito precisa
- Previsao 10 passos a frente: menos precisa
- Previsao 100 passos a frente: converge para media da serie

Isso e esperado: quanto mais longe, mais incerteza.

In [ ]:
# Simular previsao ARIMA simples (AR para simplicidade)
def forecast_ar(Y, phi, h=10):
    # Y: serie observada
    # phi: coeficientes AR
    # h: horizonte

    forecast = np.zeros(h)
    Y_extended = Y.copy()

    for step in range(h):
        # Y_t = sum(phi * Y_{t-lag})
        pred = 0
        for lag, coef in enumerate(phi):
            if len(Y_extended) > lag:
                pred += coef * Y_extended[-(lag+1)]

        forecast[step] = pred
        Y_extended = np.append(Y_extended, pred)

    return forecast

# Gerar dados e prever
np.random.seed(42)
Y_train = simulate_ar([0.7, 0.2], n_steps=100)

# Treinar (estimar phi)
phi_est = [0.7, 0.2]

# Prever
horizon = 20
forecast = forecast_ar(Y_train, phi_est, h=horizon)

# Intervalo de confianca (aproximado)
# Erro aumenta com horizonte
std_errors = np.sqrt(np.arange(1, horizon+1) * 0.5)
ci_upper = forecast + 1.96 * std_errors
ci_lower = forecast - 1.96 * std_errors

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Previsao com IC
ax = axes[0]
t_train = np.arange(len(Y_train))
t_forecast = np.arange(len(Y_train), len(Y_train) + horizon)

ax.plot(t_train, Y_train, 'b-', linewidth=2, label='Dados observados')
ax.plot(t_forecast, forecast, 'r-', linewidth=2, label='Previsao ARIMA')
ax.fill_between(t_forecast, ci_lower, ci_upper, color='red', alpha=0.2, label='IC 95%')
ax.axvline(x=len(Y_train)-1, color='k', linestyle='--', alpha=0.5)
ax.set_xlabel('Tempo')
ax.set_ylabel('Y_t')
ax.set_title('Previsao ARIMA com Intervalo de Confianca', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Erro cresce com horizonte
ax = axes[1]
ax.plot(range(1, horizon+1), std_errors, 'r-o', linewidth=2, markersize=6)
ax.set_xlabel('Horizonte (passos futuros)')
ax.set_ylabel('Desvio padrao do erro')
ax.set_title('Incerteza Aumenta com Horizonte', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.fill_between(range(1, horizon+1), 0, std_errors, alpha=0.2, color='red')

plt.tight_layout()
plt.savefig('/tmp/forecast_arima.png', dpi=100, bbox_inches='tight')
plt.show()

print("Previsao ARIMA Forward")
print("=" * 60)
print(f"Horizonte: {horizon} passos")
print()
print("Primeiras 5 previsoes:")
for i in range(min(5, horizon)):
    print(f"  Y_t+{i+1} = {forecast[i]:.3f} +/- {std_errors[i]:.3f}")
print()
print("IMPORTANTE: intervalo de confianca cresce!")
print("Previsoes distantes perdem precisao")

## 8. SARIMA: Capturando Sazonalidade

### Problema: Series com Padrao Sazonal
Exemplo: vendas aumentam no Natal, diminuem em janeiro. Ano que vem, repete.
Modelo ARIMA simples nao captura isso!

### Solucao: SARIMA (Seasonal ARIMA)
SARIMA(p,d,q)(P,D,Q,s):
- (p,d,q): parametros nao-sazonais (usua)
- (P,D,Q): parametros sazonais
- s: sazonalidade (12 para mensal, 7 para diario, etc)

Ideia: aplicar diferenciacao sazonal Y_t - Y_{t-s} para remover sazonalidade

In [ ]:
# Simular serie com sazonalidade
np.random.seed(42)
t = np.arange(0, 120)

# Trend
trend = 10 + 0.05 * t

# Sazonalidade semanal (s=7)
seasonality = 3 * np.sin(2 * np.pi * t / 7)

# AR(1)
Y = trend.copy()
for i in range(1, len(t)):
    Y = np.append(Y[:-1], Y[-1] + 0.5 * (Y[i-1] - trend[i-1]) + seasonality[i] + np.random.normal(0, 0.5))

Y = trend + seasonality + np.random.normal(0, 0.5, len(t))

# Diferenciacoes
Y_diff1 = np.diff(Y)
Y_diff_seasonal = Y[7:] - Y[:-7]  # Y_t - Y_{t-7}

fig, axes = plt.subplots(3, 2, figsize=(14, 10))

# Original
axes[0, 0].plot(Y, 'b-', linewidth=1.5)
axes[0, 0].set_title('Serie Original (com sazonalidade s=7)', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

acf_y = calculate_acf(Y, max_lag=25)
axes[0, 1].bar(range(len(acf_y)), acf_y, color='steelblue', alpha=0.7)
axes[0, 1].axvline(x=7, color='g', linestyle='--', linewidth=2, label='Sazonalidade s=7')
axes[0, 1].set_title('ACF: Picos em lag 7 (sazonalidade!)', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Diferenca 1
axes[1, 0].plot(Y_diff1, 'g-', linewidth=1.5)
axes[1, 0].set_title('Primeira Diferenca (remove trend)', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

acf_d1 = calculate_acf(Y_diff1, max_lag=25)
axes[1, 1].bar(range(len(acf_d1)), acf_d1, color='green', alpha=0.7)
axes[1, 1].axvline(x=7, color='r', linestyle='--', linewidth=2, label='Ainda ha sazonalidade!')
axes[1, 1].set_title('ACF: Ainda com sazonalidade', fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

# Diferenca sazonal
axes[2, 0].plot(Y_diff_seasonal, 'r-', linewidth=1.5)
axes[2, 0].set_title('Diferenca Sazonal: Y_t - Y_{t-7}', fontweight='bold')
axes[2, 0].grid(True, alpha=0.3)

acf_ds = calculate_acf(Y_diff_seasonal, max_lag=25)
axes[2, 1].bar(range(len(acf_ds)), acf_ds, color='red', alpha=0.7)
axes[2, 1].axhline(y=0.05, color='k', linestyle='--', alpha=0.5)
axes[2, 1].set_title('ACF: Sazonalidade removida!', fontweight='bold')
axes[2, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/sarima.png', dpi=100, bbox_inches='tight')
plt.show()

print("SARIMA: Removendo Sazonalidade")
print("=" * 60)
print("Original: ACF mostra picos em lag 7 (sazonalidade)")
print("d=1: ACF ainda mostra sazonalidade")
print("Diferenca sazonal (D=1, s=7): sazonalidade removida!")
print()
print("SARIMA(p,d,q)(P,D,Q,7) captura ambas as estruturas")

## 9. Prophet: Abordagem Diferente (Decomposicao)

### Paradigma Diferente
ARIMA: modelagem parametrica, escolher (p,d,q), estimar coeficientes
**Prophet**: decomposicao explicita em componentes interpretaveis

### Componentes Prophet
Y_t = Trend(t) + Seasonality_annual(t) + Seasonality_weekly(t) + Holiday_effect(t) + Noise(t)

Onde:
- **Trend:** crescimento linear ou logistico
- **Seasonality:** padroes repetidos (anual, semanal)
- **Holidays:** efeitos de datas especificas (Natal, Black Friday)

### Vantagens vs ARIMA
1. Lida com holidays/eventos especiais naturalmente
2. Mais robusto a outliers
3. Menos sensivel a escolha de parametros
4. Intervalos de confianca melhores

### Por que em ML
Prophet e desenvolvido pelo Facebook e e usado em producao
para prever milhoes de series (e-commerce, ads, etc).
E praticamente um black-box (nao precisa estimar (p,d,q)).

In [ ]:
# Simular decomposicao tipo Prophet
np.random.seed(42)
t = np.arange(0, 365)

# Trend: crescimento linear
trend = 20 + 0.05 * t

# Sazonalidade anual
seasonality_annual = 5 * np.sin(2 * np.pi * t / 365)

# Sazonalidade semanal
seasonality_weekly = 2 * np.sin(2 * np.pi * t / 7)

# Holidays (Natal na posicao 358)
holiday_effect = np.zeros(len(t))
holiday_positions = [80, 130, 180, 280, 358]  # alguns "holidays"
for pos in holiday_positions:
    holiday_effect[max(0, pos-3):min(len(t), pos+4)] += 3

# Noise
noise = np.random.normal(0, 0.8, len(t))

# Serie completa
Y_prophet = trend + seasonality_annual + seasonality_weekly + holiday_effect + noise

# Visualizar
fig, axes = plt.subplots(6, 1, figsize=(14, 12))

# 1. Serie original
axes[0].plot(Y_prophet, 'k-', linewidth=1)
axes[0].set_ylabel('Y_t')
axes[0].set_title('Serie Completa (Prophet)', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# 2. Trend
axes[1].plot(trend, 'b-', linewidth=2)
axes[1].set_ylabel('Trend')
axes[1].set_title('Componente Trend', fontweight='bold')
axes[1].grid(True, alpha=0.3)

# 3. Sazonalidade anual
axes[2].plot(seasonality_annual, 'g-', linewidth=1.5)
axes[2].set_ylabel('Annual')
axes[2].set_title('Sazonalidade Anual', fontweight='bold')
axes[2].grid(True, alpha=0.3)

# 4. Sazonalidade semanal
axes[3].plot(seasonality_weekly, 'orange', linewidth=1)
axes[3].set_ylabel('Weekly')
axes[3].set_title('Sazonalidade Semanal', fontweight='bold')
axes[3].grid(True, alpha=0.3)

# 5. Holiday effect
axes[4].plot(holiday_effect, 'r-', linewidth=1)
axes[4].scatter(holiday_positions, [3]*len(holiday_positions), color='red', s=100, zorder=5)
axes[4].set_ylabel('Holiday')
axes[4].set_title('Efeito de Holidays (picos especiais)', fontweight='bold')
axes[4].grid(True, alpha=0.3)

# 6. Noise
axes[5].plot(noise, 'purple', linewidth=0.5, alpha=0.7)
axes[5].set_ylabel('Noise')
axes[5].set_xlabel('Tempo (dias)')
axes[5].set_title('Ruido Residual', fontweight='bold')
axes[5].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/prophet_decomposition.png', dpi=100, bbox_inches='tight')
plt.show()

print("Prophet: Decomposicao Explicita")
print("=" * 60)
print("Y_t = Trend + Seasonality_annual + Seasonality_weekly + Holiday + Noise")
print()
print("Vantagens:")
print("1. Cada componente e interpretavel")
print("2. Facil incorporar holidays (Natal, Black Friday)")
print("3. Robusto a outliers")
print("4. Nao precisa estimar (p,d,q)!")

## 10. Comparacao: ARIMA vs Prophet vs Machine Learning

### ARIMA
**Vantagens:**
- Fundamentacao teorica solida
- Interpretavel (ACF/PACF)
- Rapido
- Bom baseline

**Desvantagens:**
- Precisa estimar (p,d,q)
- Dificil incorporar holidays
- Assume linearidade

### Prophet
**Vantagens:**
- Lida com holidays naturalmente
- Robusto a outliers
- Menos tuning necessario
- Intervalos de confianca confiáveis

**Desvantagens:**
- "Black-box" (menos interpretavel que ARIMA)
- Assume sazonalidade regular
- Requer dados mensais/semanais/diarios

### Machine Learning (LSTM, XGBoost, etc - 5D_3/5D_4)
**Vantagens:**
- Captura relacoes complexas nao-lineares
- Features exogenas facilmente
- Performance em dados complexos

**Desvantagens:**
- Precisa muito dado (>500 pontos)
- Menos interpretavel
- Mais lento para treinar
- Overfitting se nao cuidado

In [ ]:
# Comparar performance de ARIMA vs Prophet vs naive
np.random.seed(42)
n_total = 365
n_test = 50

# Gerar serie com trend + sazonalidade
t = np.arange(n_total + n_test)
trend_full = 0.05 * t
season_full = 5 * np.sin(2 * np.pi * t / 30)
noise_full = np.random.normal(0, 1, len(t))
Y_full = trend_full + season_full + noise_full

Y_train = Y_full[:n_total]
Y_test = Y_full[n_total:]

# 1. Naive: Y_t = Y_{t-1}
Y_naive = np.full(n_test, Y_train[-1])

# 2. ARIMA simulado: usar AR(1) + diferenciacao
Y_diff = np.diff(Y_train)
phi_hat = np.corrcoef(Y_diff[:-1], Y_diff[1:])[0, 1]
Y_arima = np.zeros(n_test)
last_val = Y_train[-1]
last_diff = Y_diff[-1]
for i in range(n_test):
    next_diff = phi_hat * last_diff
    Y_arima[i] = last_val + next_diff
    last_val = Y_arima[i]
    last_diff = next_diff

# 3. Prophet simulado: trend linear + sazonalidade ciclica
slope = np.polyfit(np.arange(n_total), Y_train, 1)[0]
season_cycle = Y_train[-30:]  # ultimo ciclo
Y_prophet_pred = np.zeros(n_test)
for i in range(n_test):
    Y_prophet_pred[i] = Y_train[-1] + slope * (i + 1) + season_cycle[i % 30] - np.mean(season_cycle)

# Calcular erros
mae_naive = np.mean(np.abs(Y_test - Y_naive))
mae_arima = np.mean(np.abs(Y_test - Y_arima))
mae_prophet = np.mean(np.abs(Y_test - Y_prophet_pred))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, pred, name, mae in [(axes[0], Y_naive, 'Naive', mae_naive),
                               (axes[1], Y_arima, 'ARIMA (sim)', mae_arima),
                               (axes[2], Y_prophet_pred, 'Prophet (sim)', mae_prophet)]:
    ax.plot(Y_test, 'b-', label='Real', linewidth=2)
    ax.plot(pred, 'r--', label=f'{name} (MAE={mae:.2f})')
    ax.set_title(f'{name}', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("Comparacao ARIMA vs Prophet vs Naive")
print("=" * 50)
print(f"Naive MAE:   {mae_naive:.2f}")
print(f"ARIMA MAE:   {mae_arima:.2f}")
print(f"Prophet MAE: {mae_prophet:.2f}")
print(f"\nMelhor: {'Prophet' if mae_prophet < mae_arima else 'ARIMA'}")

## 11. Exercicios Praticos

### Exercicio 1: Gerar e Analisar Serie AR(1)
Gere uma serie AR(1) com phi=0.8, calcule ACF/PACF e verifique que PACF tem 1 pico

In [ ]:
# PRATICA - Exercicio 1: AR(1) Analysis
# Gere uma serie Y com Y_t = 0.8 * Y_{t-1} + epsilon_t

# TAREFA DO ALUNO: usar simulate_ar([0.8], n_steps=150) para gerar Y
Y = None

# TAREFA DO ALUNO: calcular ACF e PACF para Y
acf = None
pacf = None

# TAREFA DO ALUNO: visualizar ACF e PACF lado a lado
# Dica: use bar plot e identifique os picos

print("Implemente o exercicio 1!")

In [ ]:
# SOLUCAO - Exercicio 1: AR(1) Analysis
Y = simulate_ar([0.8], n_steps=150)
acf = calculate_acf(Y, max_lag=15)
pacf = calculate_pacf(Y, max_lag=15)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ACF
ax = axes[0]
ax.bar(range(len(acf)), acf, color='steelblue', alpha=0.7)
ax.axhline(y=0.05, color='r', linestyle='--', alpha=0.5, label='Significancia')
ax.axhline(y=-0.05, color='r', linestyle='--', alpha=0.5)
ax.set_xlabel('Lag')
ax.set_ylabel('ACF')
ax.set_title('ACF de AR(1): Decai Exponencialmente', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# PACF
ax = axes[1]
ax.bar(range(len(pacf)), pacf, color='green', alpha=0.7)
ax.axhline(y=0.05, color='r', linestyle='--', alpha=0.5, label='Significancia')
ax.axhline(y=-0.05, color='r', linestyle='--', alpha=0.5)
ax.set_xlabel('Lag')
ax.set_ylabel('PACF')
ax.set_title('PACF de AR(1): 1 Pico Significativo!', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/ex1_ar1.png', dpi=100, bbox_inches='tight')
plt.show()

print("SOLUCAO - Exercicio 1")
print("=" * 60)
print(f"ACF decai exponencialmente (tipico de AR)")
print(f"PACF tem 1 pico significativo em lag 1 -> p=1!")
print(f"Conclusao: serie e AR(1)")

### Exercicio 2: Diferenciacao para Estacionaridade
Gere uma serie nao-estacionaria Y_t = 10 + 0.1*t + noise. Diferenciacao 1 torna estacionaria?

In [ ]:
# PRATICA - Exercicio 2: Diferenciacao
# Gere: Y_t = 10 + 0.1*t + noise

np.random.seed(42)
t = np.arange(100)

# TAREFA DO ALUNO: criar Y_t = 10 + 0.1*t + np.random.normal(0, 1, 100)
Y = None

# TAREFA DO ALUNO: calcular primeira diferenca Y_diff = np.diff(Y)
Y_diff = None

# TAREFA DO ALUNO: visualizar Y e Y_diff lado a lado
# TAREFA DO ALUNO: calcular ACF de Y e Y_diff

print("Implemente o exercicio 2!")

In [ ]:
# SOLUCAO - Exercicio 2: Diferenciacao
np.random.seed(42)
t = np.arange(100)
Y = 10 + 0.1 * t + np.random.normal(0, 1, 100)
Y_diff = np.diff(Y)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Original
ax = axes[0, 0]
ax.plot(Y, 'b-', linewidth=1.5)
ax.set_title('Serie Original (nao-estacionaria)', fontweight='bold')
ax.set_ylabel('Y_t')
ax.grid(True, alpha=0.3)

# ACF original
ax = axes[0, 1]
acf_y = calculate_acf(Y, max_lag=15)
ax.bar(range(len(acf_y)), acf_y, color='steelblue', alpha=0.7)
ax.axhline(y=0.05, color='r', linestyle='--', alpha=0.5)
ax.set_title('ACF Original: Decai MUITO LENTAMENTE', fontweight='bold')
ax.set_ylabel('ACF')
ax.grid(True, alpha=0.3, axis='y')

# Diferenciada
ax = axes[1, 0]
ax.plot(Y_diff, 'g-', linewidth=1.5)
ax.set_title('Primeira Diferenca (estacionaria!)', fontweight='bold')
ax.set_ylabel('Y_t - Y_t-1')
ax.set_xlabel('Tempo')
ax.grid(True, alpha=0.3)

# ACF diferenciada
ax = axes[1, 1]
acf_diff = calculate_acf(Y_diff, max_lag=15)
ax.bar(range(len(acf_diff)), acf_diff, color='green', alpha=0.7)
ax.axhline(y=0.05, color='r', linestyle='--', alpha=0.5)
ax.set_title('ACF Diferenciada: Decai RAPIDO', fontweight='bold')
ax.set_ylabel('ACF')
ax.set_xlabel('Lag')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/ex2_diff.png', dpi=100, bbox_inches='tight')
plt.show()

print("SOLUCAO - Exercicio 2")
print("=" * 60)
print(f"Original: ACF decai MUITO LENTAMENTE (d=0 nao funciona)")
print(f"Diferenciada: ACF decai RAPIDAMENTE (d=1 torna estacionaria!)")
print(f"Conclusao: ARIMA(p,1,q) e apropriado para serie com trend linear")

### Exercicio 3: Prever serie com Prophet (simulado)
Crie uma serie com trend + sazonalidade, faca forecast

In [ ]:
# PRATICA - Exercicio 3: Prophet-style Forecasting
# Gere: Y_t = 50 + 0.2*t + 10*sin(2*pi*t/30) + noise

np.random.seed(42)
t_train = np.arange(150)

# TAREFA DO ALUNO: criar Y_train = 50 + 0.2*t + 10*sin(2*pi*t/30) + noise
Y_train = None

# TAREFA DO ALUNO: estimar trend (reta via media dos 30 primeiros)
trend_est = None

# TAREFA DO ALUNO: estimar sazonalidade como (Y_train - trend_est)
seasonality_est = None

# TAREFA DO ALUNO: usar estas componentes para prever 30 passos afrente
h = 30
t_forecast = np.arange(150, 150 + h)
Y_forecast = None

print("Implemente o exercicio 3!")

In [ ]:
# SOLUCAO - Exercicio 3: Prophet-style Forecasting
np.random.seed(42)
t_train = np.arange(150)
trend_true = 50 + 0.2 * t_train
seasonality_true = 10 * np.sin(2 * np.pi * t_train / 30)
Y_train = trend_true + seasonality_true + np.random.normal(0, 1, len(t_train))

# Estimar trend (regressao linear simples)
coef_trend = np.polyfit(t_train, Y_train, 1)
trend_est = np.polyval(coef_trend, t_train)

# Estimar sazonalidade
seasonality_est = Y_train - trend_est

# Prever
h = 30
t_forecast = np.arange(150, 150 + h)

# Trend futuro
trend_forecast = np.polyval(coef_trend, t_forecast)

# Sazonalidade: ciclo de periodo 30
seasonality_forecast = seasonality_est[:h]

# Forecast completo
Y_forecast = trend_forecast + seasonality_forecast

fig, ax = plt.subplots(figsize=(14, 5))

# Dados
ax.plot(t_train, Y_train, 'b-', linewidth=2, label='Dados treino')
ax.plot(t_forecast, Y_forecast, 'r-', linewidth=2, label='Forecast')

# Componentes estimadas
ax.plot(t_train, trend_est, 'b--', linewidth=1, alpha=0.5, label='Trend estimado')
ax.plot(t_forecast, trend_forecast, 'r--', linewidth=1, alpha=0.5)

ax.axvline(x=150, color='k', linestyle='--', alpha=0.5)
ax.set_xlabel('Tempo')
ax.set_ylabel('Y_t')
ax.set_title('Prophet-style: Trend + Seasonality Decomposition', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/ex3_prophet.png', dpi=100, bbox_inches='tight')
plt.show()

print("SOLUCAO - Exercicio 3")
print("=" * 60)
print("Decomposicao Prophet:")
print(f"1. Estimar Trend via regressao")
print(f"2. Estimar Seasonality = Y - Trend")
print(f"3. Prever: Forecast = Trend_futuro + Seasonality_reciclada")
print()
print(f"Trend estimado: coef = {coef_trend}")
print(f"Sazonalidade ciclo: periodo = 30")

## 12. Erros Comuns na Previsao com ARIMA e Prophet

### Erro 1: Nao Diferenciar Antes de Estimar ARIMA
**Problema:** Series nao-estacionarias violam pressupostos de AR/MA.
ACF decai lentamente, parametros estimados sao ruins.

**Solucao:** Sempre verificar estacionaridade (ACF decai rapido?).
Se nao, diferenciar.

### Erro 2: Escolher (p,d,q) Visualmente Sem Validacao
**Problema:** ACF/PACF e subjetivo. Voce pode escolher (1,1,1) quando (2,1,0) e melhor.

**Solucao:** Usar Grid Search ou AIC/BIC para comparar modelos automaticamente.

### Erro 3: Horizonte Muito Longo sem Reavaliar
**Problema:** Prever 365 passos com ARIMA feito com 100 dados.
Incerteza explode, previsoes sao inúteis.

**Solucao:** Rolling forecast -- reestimar modelo a cada nova observacao.

### Erro 4: Ignorar Sazonalidade com ARIMA Comum
**Problema:** Serie com sazonalidade forte. ARIMA(1,1,1) nao captura.

**Solucao:** Usar SARIMA(p,d,q)(P,D,Q,s) ou Prophet que lida melhor.

### Erro 5: Nao Testar em Dados Separados
**Problema:** Treinar e avaliar no mesmo conjunto. Parece otimista.

**Solucao:** Time series cross-validation (train-test split preservando ordem).

In [ ]:
# Demonstrar Erros Comuns

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# Erro 1: nao diferenciar
ax = axes[0, 0]
Y_nondiff = 50 + 0.2 * t_train + np.random.normal(0, 1, len(t_train))
acf_nondiff = calculate_acf(Y_nondiff, max_lag=15)
ax.bar(range(len(acf_nondiff)), acf_nondiff, color='red', alpha=0.7)
ax.axhline(y=0.05, color='k', linestyle='--', alpha=0.5)
ax.set_title('ERRO 1: ACF Nao-Estacionaria\n(decai muito lento!)', fontweight='bold')
ax.set_ylabel('ACF')
ax.grid(True, alpha=0.3, axis='y')

# Solucao: diferenciar
ax = axes[0, 1]
Y_diff = np.diff(Y_nondiff)
acf_diff = calculate_acf(Y_diff, max_lag=15)
ax.bar(range(len(acf_diff)), acf_diff, color='green', alpha=0.7)
ax.axhline(y=0.05, color='k', linestyle='--', alpha=0.5)
ax.set_title('SOLUCAO 1: ACF Estacionaria\n(decai rapido!)', fontweight='bold')
ax.set_ylabel('ACF')
ax.grid(True, alpha=0.3, axis='y')

# Erro 3: horizonte longo
ax = axes[0, 2]
horizons = [1, 5, 10, 20, 50, 100]
forecasts = []
errors = []
for h in horizons:
    # Previsao media
    f = np.mean(Y_train) * np.ones(h)
    forecasts.append(f)
    # Erro cresce com h
    err = np.std(Y_train) * np.sqrt(h)
    errors.append(err)

ax.plot(horizons, errors, 'r-o', linewidth=2, markersize=8)
ax.set_xlabel('Horizonte (passos)')
ax.set_ylabel('Intervalo de Confianca (95%)')
ax.set_title('ERRO 3: Incerteza Explode\ncom Horizonte Longo', fontweight='bold')
ax.grid(True, alpha=0.3)

# Erro 4: ignorar sazonalidade
ax = axes[1, 0]
Y_seasonal = 50 + 10 * np.sin(2 * np.pi * t_train / 30) + np.random.normal(0, 1, len(t_train))
acf_seasonal = calculate_acf(Y_seasonal, max_lag=35)
ax.bar(range(len(acf_seasonal)), acf_seasonal, color='red', alpha=0.7)
ax.axvline(x=30, color='g', linestyle='--', linewidth=2, label='Sazonalidade')
ax.set_title('ERRO 4: Ignorar Sazonalidade\n(picos em lag 30!)', fontweight='bold')
ax.set_ylabel('ACF')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Solucao: usar SARIMA
ax = axes[1, 1]
Y_diff_seasonal = Y_seasonal[30:] - Y_seasonal[:-30]
acf_diff_seasonal = calculate_acf(Y_diff_seasonal, max_lag=35)
ax.bar(range(len(acf_diff_seasonal)), acf_diff_seasonal, color='green', alpha=0.7)
ax.axhline(y=0.05, color='k', linestyle='--', alpha=0.5)
ax.set_title('SOLUCAO 4: Diferenca Sazonal\n(sazonalidade removida!)', fontweight='bold')
ax.set_ylabel('ACF')
ax.grid(True, alpha=0.3, axis='y')

# Erro 5: train/test split errado
ax = axes[1, 2]
# Simulacao otimista (treino = teste)
train_size = 100
train_error = 0.8
test_on_train = 0.9  # otimista (mesmo conjunto!)
test_on_holdout = 0.6  # realista (dados novos)

methods = ['ARIMA\n(train/test\nenganoso)', 'ARIMA\n(validacao\ncorreta)']
errors_comparison = [test_on_train, test_on_holdout]
colors_err = ['red', 'green']

bars = ax.bar(methods, errors_comparison, color=colors_err, alpha=0.7)
ax.set_ylabel('Acuracia (R2)')
ax.set_ylim([0, 1])
ax.set_title('ERRO 5: Validacao Enganosa\nvs Correta', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, err in zip(bars, errors_comparison):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{err:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/common_errors.png', dpi=100, bbox_inches='tight')
plt.show()

print("Erros Comuns em Previsao com ARIMA")
print("=" * 60)
print("ERRO 1: Nao diferenciar antes de ARIMA")
print("  -> Solucao: Verificar estacionaridade com ACF")
print()
print("ERRO 2: Escolher (p,d,q) visualmente")
print("  -> Solucao: Usar grid search + AIC/BIC")
print()
print("ERRO 3: Horizonte muito longo")
print("  -> Solucao: Rolling forecast, reestimar periodicamente")
print()
print("ERRO 4: Ignorar sazonalidade")
print("  -> Solucao: SARIMA ou Prophet")
print()
print("ERRO 5: Train/test split errado")
print("  -> Solucao: Time series cross-validation (preservar ordem)")

### O que observar sobre Correlacao Temporal

Em series temporais, observacoes proximas sao CORRELACIONADAS.
Isso viola o pressuposto de independencia de modelos estatisticos simples.

Por isso: validacao com train/test split ALEATORIO nao funciona.
Necessario preservar ordem temporal (time series cross-validation).

Exemplo: se Y_1=10, Y_2=12, entao Y_3 provavelmente ~14 (continue a tendencia)
Modelos que exploram esta correlacao (AR, ARIMA) sao mais acurados.

### O que observar sobre ACF/PACF como Ferramentas Diagnosticas

ACF (autocorrelacao) mostra quão forte é a correlacao entre Y_t e Y_{t-k}.
PACF (autocorrelacao parcial) remove correlacoes intermediarias.

Lendo ACF/PACF:
- Picos grandes fora da banda de confianca (vermelho) = estrutura presente
- ACF que decai lentamente = serie nao-estacionaria (precisa d)
- PACF com p picos = AR(p) pode funcionar
- ACF com q picos = MA(q) pode funcionar

Dica: sempre gerar ACF/PACF antes de escolher (p,d,q).

### O que observar sobre Raizes Unitarias

Uma serie tem RAIZ UNITARIA se a equacao caracteristica Y_t = phi*Y_{t-1} tem phi=1.
Isso significa a serie e I(1) = nao-estacionaria = precisa diferenciar.

Teste de Dickey-Fuller (ADF): hipotese nula = raiz unitaria presente
Se p-value < 0.05: rejeita H0, serie e estacionaria (d=0)
Se p-value >= 0.05: nao rejeita, serie e nao-estacionaria (d=1 ou mais)

Sem testar raiz unitaria, pode escolher d errado (underfitting ou overfitting).

### O que observar sobre Intervalo de Confianca em Horizonte Longo

Intervalo de confianca de ARIMA cresce quadraticamente com horizonte.
Apos ~30-50 passos, IC pode englobar TAREFA DO ALUNO o range historico (inútil).

Isso e ESPERADO e CORRETO:
- Informacao diminui com distancia temporal
- Sem features exogenas, incerteza aumenta
- Prophet tem ICs mais realistas que ARIMA

Nunca confie em previsoes muito longas sem features exogenas!

### O que observar sobre a Importancia de Estacionaridade

Estacionaridade e FUNDAMENTAL para ARIMA funcionar. Uma serie nao-estacionaria
viola os pressupostos matematicos (erros nao sao iid, variancia muda, etc).

Sinais de nao-estacionaridade:
1. Media muda (trend)
2. Variancia muda
3. ACF decai muito lentamente
4. Raiz unitaria (teste de Dickey-Fuller)

### O que concluir sobre o Numero de Diferenciacoes Necessarias

Quantas vezes diferenciar?
- Teste Dickey-Fuller: se p > 0.05, precisa diferenciar (d++)
- Regra empirica: maioria das series precisa d=1
- Raras series precisam d=2 (series de segunda ordem)
- d >= 3 e suspeito (provavelmente overfitting)

OBS: diferenciar demais REMOVE informacao util!
Melhor: comece com d=1, teste se ACF fica estacionario
Se nao, incremente para d=2 (raramente necessario)

### O que concluir sobre ARIMA vs Prophet

ARIMA: paradigma PARAMETRICO (escolher p,d,q, estimar coeficientes)
Prophet: paradigma DECOMPOSICIONAL (trend + seasonality + holidays)

Para serie "simples" (trend linear, sazonalidade regular): ARIMA melhor
Para serie "complexa" (holidays, mudancas abruptas): Prophet melhor

### O que concluir sobre a Qualidade da Previsao

Nunca confie em 1 numero (ex: RMSE = 5). Sempre:
1. Calcule intervalo de confianca (95%)
2. Compare com baseline simples (media, seasonal naive)
3. Teste em multiplos horizontes (1, 7, 30 passos)
4. Valide em dados FORA da amostra (time series CV)
5. Acompanhe performance em producao (model monitoring)

Previsoes que se degradam no tempo = dados mudaram, modelo precisa reavaliar.

### Por que em ML (Producao e Negocio)
Em producao, previsoes devem ser:
1. **Estáveis:** intervalos de confianca nao explodem
2. **Calibradas:** quando digo 95%, acerta 95%
3. **Updateaveis:** modelo recebe dados novos, refaz previsao
4. **Explicáveis:** CEO pergunta "por que previu X?" e voce responde

ARIMA/Prophet: bons nisso. Deep Learning: piores (black-box, instavel).

### Conexao com Notebooks sobre Series Temporais

5D_1 (Analise Temporal): ACF/PACF para entender estrutura temporal
5D_2 (Previsao Classica - este notebook): ARIMA, Prophet, SARIMA metodos classicos
5D_3 (Deep Learning Temporal): LSTM, GRU, Transformer para series complexas
5D_4 (Ensemble/Hybrid Temporal): combinar ARIMA + ML para melhor acuracia
4_1, 4_2 (Regularizacao): AIC/BIC sao formas de regularizacao modelo
3_1 (Hyperparameter Tuning): Grid Search em (p,d,q) similar a hyperopt em DNNs

### O que observar sobre Interpretabilidade

ARIMA e mais interpretavel que Prophet:
- Parametros phi, theta tem significado exato (quanto cada lag importa)
- ACF/PACF revelam estrutura oculta da serie
- Pode explicar para stakeholders por que preve o que preve

Prophet e menos interpretavel mas mais robusto:
- Componentes sao "black-box" (tendencias curvilíneas, parametros de holidays)
- Mas intervalos de confianca sao mais realistas
- Melhor para Series com comportamento irregular

### O que concluir sobre a Escolha do Horizonte

Quanto mais longe voce quer prever:
- 1-7 passos: ARIMA/Prophet sao excelentes (acuracia alta)
- 8-30 passos: ARIMA/Prophet declinam, ML comeca a ganhar
- 30+ passos: Necessario incorporar features exogenas (economia, marketing, clima)

Sem features exogenas, horizonte longo sempre converge para media da serie.

### Conexao com Notebooks sobre Validacao e Metricas

4_3 (Interpretabilidade): entender componentes ARIMA de forma visual
2_1, 2_2 (Metricas): MAE, RMSE, MAPE aplicados a series (nao mais CV aleatoria!)

Time series cross-validation (deixar de fora ultimos k passos, medir erro)
e o padrao ouro DIFERENTE de CV aleatoria (3_2). Ordem temporal IMPORTA.

Metricas tipicas:
- RMSE (root mean squared error): penaliza erros grandes
- MAE (mean absolute error): metrica robusta
- MAPE (mean absolute % error): percentual, independente escala

### O que observar sobre Sazonalidade

Sazonalidade pode ser:
- ADITIVA: Y = Trend + Seasonal (usar ARIMA/Prophet standard)
- MULTIPLICATIVA: Y = Trend * Seasonal (log-transform, depois ARIMA)

Series com grande trend e sazonalidade forte: MULTIPLICATIVA melhor
Dica: plote scatter plot Trend vs Seasonal para visualizar tipo

### O que observar sobre Invertibilidade e Causalidade

Em ARIMA: nem todo (p,d,q) e valido!
- AR deve ser CAUSAL (phi_i dentro do circulo unitario)
- MA deve ser INVERTIVEL (theta_i dentro do circulo unitario)

Se violar, raizes do polinomio caracteristico expldem, previsoes divergem

### O que concluir sobre Features Exogenas

ARIMAX = ARIMA + features exogenas (temperatura, preco do competidor, etc)
Quando voce TEM features correlacionadas, ARIMAX supera ARIMA puro

Mas: cuidado com data leakage! Features futuras nao sao conhecidas!

### Conexao com Notebooks sobre Hyperparameter Tuning

Grid search em (p,d,q) e tedioso. Auto ARIMA (equivalente a Hyperopt em 3_1)
busca (p,d,q) automaticamente usando AIC/BIC (criterios de selecao modelo)

Algoritmo: comeca com (0,0,0), testa vizinhos, para quando AIC nao melhora
Conexao com 5B_5: Auto-ARIMA e exemplo de AutoML temporal.

### O que observar sobre Overfitting em Horizonte Longo

ARIMA com muitos parametros (p,q altos) pode OVERFIT no historico
Resultado: bom fit no treino, mas previsoes divergem para horizonte longo

Solucao: usar AIC/BIC que penalizam numero de parametros

### O que concluir sobre a Importancia da Decomposicao

Prophet e popular porque DECOMPOSICAO e intuitiva para negocio:
- CEO quer saber: quanto vem de trend vs seasonality?
- Seasonality semanal vs anual?
- Holiday effect (quanto Black Friday aumenta vendas?)

ARIMA nao oferece essa visao facil (parametros phi, theta nao sao intuitivos)

### O que observar sobre Outliers e Robustez

ARIMA e sensivel a outliers (quebram ACF/PACF)
Prophet e robusto por construcao (decomposicao absorve anomalias)

Se serie tem outliers:
- ARIMA: limpar outliers, depois estimar
- Prophet: robusto direto (intervalos de confianca podem avisar)

### Conexao com Notebooks sobre Regularizacao

Diferenciacao (d em ARIMA) e forma de REGULARIZACAO:
- Reduz complexidade (menos graus de liberdade)
- Pena por nao-estacionaridade
- Similar a regularizacao L1/L2 em redes neurais (4_1)

### O que concluir sobre Selection de Modelos

Grid Search em (p,d,q) e tedioso manualmente, mas necessario.
Criterios de selecao:
- **AIC:** Akaike Information Criterion (penaliza parametros)
- **BIC:** Bayesian IC (penaliza mais parametros que AIC)
- **RMSE:** raiz do erro quadratico medio (sem penalizar parametros)

Auto-ARIMA automatiza isso: testa (0,0,0), depois incrementa p,d,q
parandoquando AIC deixa de melhorar.

### O que concluir sobre Ensemble de Modelos

Nenhum modelo e perfeito para TAREFA DO ALUNOS os dados.
Melhor pratica: combinar previsoes de ARIMA + Prophet + ML:
- ARIMA captura autocorrelacao linear
- Prophet captura trend + seasonality
- ML captura padroes complexos nao-lineares

Weighted average baseado em performance historica supera qualquer modelo individual.

### Conexao com Notebooks sobre Deep Learning

5D_3 (LSTM): modelos autoregressivos neurais
- Y_t = f(Y_{t-1}, Y_{t-2}, ..., Y_{t-p})
- f e uma rede neural (nao combinacao linear como ARIMA)
- Captura relacoes nao-lineares
- Precisa de MUITO dado (>500-1000 amostras)

Comparacao:
- ARIMA: 50+ dados, linear, rapido
- LSTM: 500+ dados, nao-linear, lento

### O que concluir sobre Transformacao de Series

Series com VARIANCIA nao-constante deve ser transformada:
- Log-transform: Y' = log(Y) estabiliza variancia
- Box-Cox: generalizacao de log-transform, encontra transformacao otima
- Diferenca de logs: Y' = log(Y_t) - log(Y_{t-1}) = retorno percentual

Apos ARIMA em Y', fazer transformacao inversa para prever em escala original.

### O que concluir sobre a Escolha entre ARIMA e Prophet

**Escolha ARIMA se:**
- Serie e "simples" (trend linear, sazonalidade regular, sem holidays)
- Quer interpretabilidade maxima (ACF/PACF, parametros)
- Poucos dados (<200 observacoes)

**Escolha Prophet se:**
- Serie tem holidays/eventos (Black Friday, Natal, etc)
- Tem mudancas abruptas (lockdown, mudanca de politica)
- Quer intervalos de confianca realistas
- Series nao-estacionarias complexas

Dica: teste ambos, compare em validacao, escolha o melhor!

### Conexao com Notebooks sobre Comparacao com Baselines

2_2 (Metricas e Avaliacao): sempre comparar com baselines:
- Media historica
- Seasonal naive (Y_t = Y_{t-s})
- Drift (regressao linear no tempo)

Se ARIMA/Prophet nao supera estes, volta ao desenho.

### Conexao com Notebooks sobre Features Exogenas e ARIMAX

5D_3, 5D_4 (Deep Learning + ML): quando ARIMAX?
- Precisa features exogenas correlacionadas
- Exemplo: prever vendas (Y) usando preco (X1), temperatura (X2)
- ARIMAX = ARIMA + regressao linear em X

Mas: Deep Learning (LSTM) captura padroes nao-lineares entre Y e X melhor.
Tradeoff: ARIMAX interpretavel, LSTM precisa mais dados.

### Conexao com Notebooks sobre Interpretabilidade e Explicabilidade

4_3 (SHAP, LIME): interpretabilidade de modelos ML
Similar em ARIMA:
- Parametros phi = "quanto passado importa"
- Parametros theta = "quanto erros passados importam"
- ACF/PACF = "onde esta a estrutura"

Prophet e menos interpretavel (parametros nao tem significado direto)
Mas: decomposicao explicita (Trend, Seasonality) e mais intuitiva que phi/theta.

### Conexao com Notebooks sobre Economia e Dados

5A_2 (Transfer Learning): ARIMA em base de dados nova:
- Pode reusar conhecimento de series anteriores?
- Exemplo: prever vendas em nova filial com dados limitados
- Resposta: nao direto (parametros phi nao transferem)
- Melhor: usar dados historicos de filiais similares

Deep Learning (5D_3) TRANSFERE melhor: embeddings aprendem features genericas.

### Proximos Passos

No proximo notebook (5D_3), usaremos DEEP LEARNING (LSTM, Transformers)
para capturar relacoes nao-lineares complexas, mas com custo de dados maiores.
A questao: "quando usar ARIMA vs Deep Learning" sera respondida.

## Resumo e Proximos Passos

### Hierarquia de Conceitos

```
Previsao de Series Temporais
  |
  +-- Abordagem Classica (Parametrica)
  |     |
  |     +-- AR(p): Y_t depende de valores passados
  |     |     |-- ACF decai, PACF corta em p
  |     |     +-- Bom para series persistentes
  |     |
  |     +-- MA(q): Y_t depende de erros passados
  |     |     |-- ACF corta em q, PACF decai
  |     |     +-- Bom para capturar shocks
  |     |
  |     +-- ARMA(p,q): combinacao AR + MA
  |     |     +-- Flexivel, maioria das series
  |     |
  |     +-- ARIMA(p,d,q): adiciona diferenciacao
  |     |     |-- d diferenciacoes para estacionaridade
  |     |     +-- PADRÃO para series nao-estacionarias
  |     |
  |     +-- SARIMA(p,d,q)(P,D,Q,s): adiciona sazonalidade
  |           +-- Para series com padrao sazonal regular
  |
  +-- Abordagem Decomposicional (Prophet)
  |     |
  |     +-- Trend: crescimento linear/logistico
  |     +-- Seasonality: padroes repetidos
  |     +-- Holidays: efeitos de eventos especiais
  |     +-- Noise: residual aleatorio
  |
  +-- Deep Learning (proximos notebooks)
        +-- LSTM, GRU, Transformer
        +-- Captura relacoes nao-lineares
        +-- Requer mais dados (>500 pontos)
```

### Conectando Conceitos

| Conceito | ARIMA | Prophet | Deep Learning |
|----------|-------|---------|---------------|
| Estacionaridade | CRITICO | Nao precisa | Nao precisa |
| Parametros | (p,d,q) | poucos | muitos |
| Dados necessarios | 50+ | 100+ | 500+ |
| Interpretabilidade | ALTA | MEDIA | BAIXA |
| Holidays | Dificil | FACIL | So com features |
| Velocidade | RAPIDA | Rapida | Lenta |
| Capacidade nao-linear | Limitada | Limitada | ALTA |

### Checklist de Competencias

- [ ] Entendo a diferenca entre AR, MA, ARMA
- [ ] Consigo ler ACF e PACF para escolher (p,q)
- [ ] Sei quando (e como) diferenciar uma serie
- [ ] Consigo implementar ARIMA simples com numpy
- [ ] Entendo SARIMA e para que sazonalidade serve
- [ ] Consigo descrever os componentes de Prophet
- [ ] Sei quando usar ARIMA vs Prophet na pratica
- [ ] Entendo as armadilhas (horizonte, validacao, etc)

### Proximos Passos

**Notebook 5D_3:** Deep Learning para Series (LSTM)
- RNNs e Transformers para capturar padroes complexos
- Quando deixar ARIMA/Prophet e usar Deep Learning
- Comparacao com MetaLearners (AutoML para series)

**Notebook 5D_4:** Ensemble e Hybrid Models
- Combinar ARIMA + Deep Learning
- Weighted averaging baseado em historico de performance
- Abordagem moderna para maximizar acuracia

**Pratica:** Aplicar em dados reais
- E-commerce: prever vendas mensais
- Financas: prever precos/volatilidade
- Energia: prever consumo de eletricidade
- Saude: prever casos de epidemias